# 08 — Robustness invariants and end-to-end computational cost

This notebook is **post-freeze**. It does not tune the encoder, payload levels, local-risk model, allocator weight, detector, or inclusion rules.

It answers three separate questions:

1. **Strict reversibility and lossless-storage robustness.** Did every feasible frozen-test case recover the original pixel array and message exactly, including the PNG round-trip checks already performed in notebook 06?
2. **Payload accounting.** How much of the gross embedded payload is consumed by the compact side-information accounting model?
3. **Computational cost.** What is the cost of *native* strategy-specific allocation plus block analysis, embedding, and extraction? In particular, the expensive SRM-derived local-risk computation is charged only to `detectability` and `joint`; `raster`, `random`, and `predictability` are not charged for a detector they do not need.

The primary timing benchmark uses a deterministic subset of the frozen test split at the already frozen diagnostic payload of **0.009 net bpp**. This is a performance characterization only; test results are not used to modify the method.

> **Scope of robustness.** This is strict reversible data hiding, not a robust watermarking channel. JPEG recompression, filtering, resizing, or noise applied *after embedding* are not claimed to preserve exact recovery. We therefore do not redefine success by destructive-channel survival. Optional cross-QF analysis at the end tests **source-preprocessing transfer before embedding**, not post-embedding attack robustness.


In [ ]:
from pathlib import Path
from time import perf_counter_ns
import json, joblib, yaml, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import psutil

from rdhlab.io import read_gray
from rdhlab.blockcodec import analyze_blocks
from rdhlab.pipeline import run_frozen_image_precomputed
from rdhlab.freeze_protocol import sha256_file, stable_id_hash, atomic_write_json
from rdhlab.final_steganalysis import validate_test_completion
from rdhlab.resources import build_strategy_order, sideinfo_ratios, summarize_exact_recovery, run_with_peak_rss

config=yaml.safe_load(Path('/workspace/config/experiment.yaml').read_text())
seed=int(config['project']['seed']); bs=int(config['dataset']['block_size'])
manifest_path=Path(config['dataset']['prepared_manifest'])
manifest=pd.read_csv(manifest_path)
test=manifest[manifest.split=='test'].reset_index(drop=True)
test['source_id']=test.source_id.astype(str)

allocator_path=Path('/workspace/config/frozen_allocator.json')
allocator=json.loads(allocator_path.read_text())
alpha=float(allocator['alpha']); payloads=list(map(float,allocator['payload_levels']))
primary_bpp=float(allocator['teacher_payload_bpp'])
assert np.isclose(alpha,0.25)
assert np.isclose(primary_bpp,0.009)

risk_path=Path('/workspace/results/models/srm_teacher_local_risk.joblib')
local_risk=joblib.load(risk_path)

out06=Path('/workspace/results/frozen_test_final')
complete=json.loads((out06/'test_run_complete.json').read_text())
validate_test_completion(complete,expected_cases=40000)
per06=pd.read_csv(out06/'per_image.csv')
per06['source_id']=per06.source_id.astype(str)
common=pd.read_csv(out06/'common_feasible_ids.csv')
common['source_id']=common.source_id.astype(str)
protocol06=json.loads((out06/'test_protocol.json').read_text())
context_dir=out06/'contexts'/protocol06['context_tag']

# Provenance checks.
if sha256_file(risk_path)!=allocator['provenance_sha256']['srm_teacher_local_risk_joblib']:
    raise RuntimeError('Frozen local-risk model hash mismatch.')
if stable_id_hash(test.source_id.tolist())!=complete['test_source_ids_sha256']:
    raise RuntimeError('Test source-ID hash mismatch.')

out=Path('/workspace/results/robustness_resources')
out.mkdir(parents=True,exist_ok=True)

print('alpha =',alpha)
print('payloads =',payloads)
print('primary benchmark payload =',primary_bpp)
print('06 status =',complete['status'])
print('No retuning is permitted.')


## 1. Exact reversibility, PNG round-trip checks, and side-information overhead

In [ ]:
# Re-check the frozen 06 invariants from its per-case table.
recovery=summarize_exact_recovery(per06)
assert recovery['exact_image_feasible']==recovery['feasible_cases']
assert recovery['exact_message_feasible']==recovery['feasible_cases']
assert recovery['zero_ber_feasible']==recovery['feasible_cases']
assert recovery['file_io_exact_image']==recovery['file_io_feasible_cases']
assert recovery['file_io_exact_message']==recovery['file_io_feasible_cases']

recovery.update({
    'metadata_physically_embedded':False,
    'side_information_status':'accounted compact binary model; metadata supplied externally to decoder in v0.2.0',
    'post_embedding_lossy_channel_robustness_claimed':False,
})
atomic_write_json(out/'reversibility_invariants.json',recovery)
print(json.dumps(recovery,indent=2))

ok=per06[per06.feasible.astype(bool)].copy()
rat=ok.apply(lambda r: sideinfo_ratios(r.gross_payload_bits,r.sideinfo_bits,r.net_payload_bits),axis=1,result_type='expand')
ok=pd.concat([ok.reset_index(drop=True),rat.reset_index(drop=True)],axis=1)
ok['sideinfo_bpp']=ok.sideinfo_bits.astype(float)/(256.0*256.0)

side_summary=(ok.groupby(['strategy','target_net_bpp'],as_index=False)
    .agg(
        n=('source_id','size'),
        gross_payload_bits_mean=('gross_payload_bits','mean'),
        net_payload_bits_mean=('net_payload_bits','mean'),
        sideinfo_bits_mean=('sideinfo_bits','mean'),
        sideinfo_bits_median=('sideinfo_bits','median'),
        sideinfo_bpp_mean=('sideinfo_bpp','mean'),
        sideinfo_fraction_gross_mean=('sideinfo_fraction_gross','mean'),
        sideinfo_fraction_gross_median=('sideinfo_fraction_gross','median'),
        sideinfo_per_net_mean=('sideinfo_per_net','mean'),
        used_blocks_mean=('used_blocks','mean'),
    ))
side_summary.to_csv(out/'sideinfo_summary.csv',index=False)
display(side_summary)


In [ ]:
fig,ax=plt.subplots(figsize=(6.6,4.5))
for strategy in config['allocator']['strategies']:
    z=side_summary[side_summary.strategy==strategy].sort_values('target_net_bpp')
    ax.plot(z.target_net_bpp,100*z.sideinfo_fraction_gross_mean,marker='o',label=strategy)
ax.set_xlabel('Net payload (bpp)')
ax.set_ylabel('Mean side information / gross payload (%)')
ax.set_title('Accounted reversible side-information overhead')
ax.grid(True,alpha=.2); ax.legend(); fig.tight_layout()
fig.savefig(out/'sideinfo_overhead_vs_payload.png',dpi=300)
plt.show()


## 2. Native strategy-specific computational cost

The old `encode_ms` column measures only the blockwise embedding loop after allocation context and block plans already exist. Here the sender-side cost is reconstructed more completely:

\[
T_{\rm sender}=T_{\rm order}+T_{\rm block\ analysis}+T_{\rm embed}.
\]

For `detectability` and `joint`, `T_order` includes all SRM-teacher local probes required to obtain \(D_i\). For `predictability`, it includes only \(P_i\). `raster` and `random` use no detector. The extraction cost is reported separately as \(T_{\rm decode}\).

The benchmark does **not** include dataset loading in `sender_compute_ms`; image read time is recorded separately. It also records the wall time of `run_frozen_image_precomputed`, which includes extraction and metric calculation and is therefore useful as a reproducibility-oriented pipeline measure rather than a pure codec time.

In [ ]:
strategies=list(config['allocator']['strategies'])
ids_primary=common[np.isclose(common.target_net_bpp.astype(float),primary_bpp)].source_id.tolist()
idset=set(ids_primary)
bench_test=test[test.source_id.isin(idset)].copy()
# Preserve frozen manifest order; deterministic and independent of timing values.
BENCH_N=min(100,len(bench_test))
bench_test=bench_test.head(BENCH_N).reset_index(drop=True)
print('Timing benchmark images:',BENCH_N)
print('Cases:',BENCH_N*len(strategies))

# Small warm-up not included in output.
wx=read_gray(bench_test.iloc[0].path); wsid=str(bench_test.iloc[0].source_id)
_ = analyze_blocks(wx,bs)
_ = build_strategy_order(wx,wsid,'predictability',local_risk,alpha,bs,seed)

rows=[]
proc=psutil.Process(os.getpid())
for i,row in bench_test.iterrows():
    sid=str(row.source_id)
    t0=perf_counter_ns(); x=read_gray(row.path); read_ms=(perf_counter_ns()-t0)/1e6

    # Cache from 06 is used only as a correctness oracle, never for timing.
    ctx=joblib.load(context_dir/f'{sid}.joblib')
    frozen_orders=ctx['orders']

    for strategy in strategies:
        rss_before=proc.memory_info().rss/1024**2

        order,block_rows,order_ms=build_strategy_order(x,sid,strategy,local_risk,alpha,bs,seed)
        if not np.array_equal(order,np.asarray(frozen_orders[strategy],dtype=int)):
            raise RuntimeError(f'Native order differs from frozen 06 order: {sid} {strategy}')

        t1=perf_counter_ns(); plans=analyze_blocks(x,bs); plan_ms=(perf_counter_ns()-t1)/1e6
        t2=perf_counter_ns()
        rr=run_frozen_image_precomputed(
            x,sid,primary_bpp,strategy,{strategy:order},block_rows,bs,seed,
            False,None,plans=plans
        )
        pipeline_after_context_ms=(perf_counter_ns()-t2)/1e6
        if not rr['feasible']:
            raise RuntimeError(f'Benchmark common-feasible case became infeasible: {sid} {strategy}')
        if not (rr['exact_image'] and rr['exact_message'] and float(rr['ber'])==0.0):
            raise RuntimeError(f'Benchmark reversibility failure: {sid} {strategy}')

        old=per06[(per06.source_id==sid)&(per06.strategy==strategy)&np.isclose(per06.target_net_bpp.astype(float),primary_bpp)]
        if len(old)!=1 or not bool(old.iloc[0].feasible):
            raise RuntimeError(f'Missing frozen 06 reference: {sid} {strategy}')
        if not np.isclose(float(rr['psnr']),float(old.iloc[0].psnr),rtol=0,atol=1e-10):
            raise RuntimeError(f'Benchmark output differs from 06: {sid} {strategy}')

        rss_after=proc.memory_info().rss/1024**2
        sender_ms=float(order_ms+plan_ms+rr['encode_ms'])
        rows.append({
            'source_id':sid,'strategy':strategy,'target_net_bpp':primary_bpp,
            'read_ms':float(read_ms),'order_ms':float(order_ms),'block_analysis_ms':float(plan_ms),
            'encode_ms':float(rr['encode_ms']),'decode_ms':float(rr['decode_ms']),
            'sender_compute_ms':sender_ms,
            'codec_roundtrip_compute_ms':float(sender_ms+rr['decode_ms']),
            'pipeline_after_context_wall_ms':float(pipeline_after_context_ms),
            'rss_before_mib':float(rss_before),'rss_after_mib':float(rss_after),
            'rss_boundary_delta_mib':float(rss_after-rss_before),
            'used_blocks':int(rr['used_blocks']),
        })
    if (i+1)%10==0 or i+1==BENCH_N:
        print('timing',i+1,'/',BENCH_N)

timing=pd.DataFrame(rows)
timing.to_csv(out/'end_to_end_timing_per_case.csv',index=False)

q=lambda s,p: float(np.percentile(s,p))
timing_summary=(timing.groupby('strategy',as_index=False)
    .agg(
        n=('source_id','size'),
        order_ms_mean=('order_ms','mean'),order_ms_median=('order_ms','median'),
        block_analysis_ms_mean=('block_analysis_ms','mean'),block_analysis_ms_median=('block_analysis_ms','median'),
        encode_ms_mean=('encode_ms','mean'),encode_ms_median=('encode_ms','median'),
        decode_ms_mean=('decode_ms','mean'),decode_ms_median=('decode_ms','median'),
        sender_compute_ms_mean=('sender_compute_ms','mean'),sender_compute_ms_median=('sender_compute_ms','median'),
        roundtrip_compute_ms_mean=('codec_roundtrip_compute_ms','mean'),roundtrip_compute_ms_median=('codec_roundtrip_compute_ms','median'),
        rss_boundary_delta_mib_mean=('rss_boundary_delta_mib','mean'),
    ))
# Add distribution tails explicitly.
for col in ['order_ms','sender_compute_ms','codec_roundtrip_compute_ms']:
    p95=timing.groupby('strategy')[col].quantile(.95).rename(col+'_p95').reset_index()
    timing_summary=timing_summary.merge(p95,on='strategy')
timing_summary.to_csv(out/'end_to_end_timing_summary.csv',index=False)
display(timing_summary)


In [ ]:
# Coarse observed RSS peak benchmark on a much smaller deterministic subset.
# Polling is kept out of the timing benchmark because it perturbs wall time.
MEM_N=min(20,len(bench_test))
mem_rows=[]
for strategy in strategies:
    for _,row in bench_test.head(MEM_N).iterrows():
        sid=str(row.source_id); x=read_gray(row.path)
        def work():
            order,br,_=build_strategy_order(x,sid,strategy,local_risk,alpha,bs,seed)
            plans=analyze_blocks(x,bs)
            return run_frozen_image_precomputed(x,sid,primary_bpp,strategy,{strategy:order},br,bs,seed,False,None,plans=plans)
        rr,mm=run_with_peak_rss(work,poll_s=.005)
        if not rr['feasible'] or not rr['exact_image'] or not rr['exact_message']:
            raise RuntimeError('Memory benchmark invariant failed.')
        mem_rows.append({'source_id':sid,'strategy':strategy,**mm})

memory=pd.DataFrame(mem_rows)
memory.to_csv(out/'observed_rss_per_case.csv',index=False)
mem_summary=(memory.groupby('strategy',as_index=False)
    .agg(n=('source_id','size'),
         observed_peak_mib_median=('rss_observed_peak_mib','median'),
         observed_delta_mib_median=('rss_observed_delta_mib','median'),
         observed_delta_mib_p95=('rss_observed_delta_mib',lambda x:float(np.percentile(x,95)))))
mem_summary.to_csv(out/'observed_rss_summary.csv',index=False)
display(mem_summary)
print('Note: RSS values are polling-based observed peaks, not exact allocator-level maxima.')


In [ ]:
fig,ax=plt.subplots(figsize=(6.6,4.5))
z=timing_summary.set_index('strategy').loc[strategies]
ax.bar(z.index,z.sender_compute_ms_median)
ax.set_ylabel('Median sender compute time (ms)')
ax.set_title('Native allocation + block analysis + embedding')
ax.tick_params(axis='x',rotation=25)
fig.tight_layout(); fig.savefig(out/'sender_compute_time_by_strategy.png',dpi=300); plt.show()

fig,ax=plt.subplots(figsize=(6.6,4.5))
for strategy in strategies:
    zz=timing[timing.strategy==strategy]
    ax.scatter(zz.order_ms,zz.encode_ms,s=12,alpha=.45,label=strategy)
ax.set_xlabel('Allocation/order time (ms)')
ax.set_ylabel('Embedding time (ms)')
ax.set_title('Allocation dominates codec cost where local risk is used')
ax.grid(True,alpha=.2); ax.legend(); fig.tight_layout()
fig.savefig(out/'allocation_vs_embedding_time.png',dpi=300); plt.show()


## 3. Optional source-preprocessing transfer (QF75/QF85/QF95)

This experiment is **secondary** and disabled by default. If enabled, it applies the already frozen encoder/risk model/\(\alpha\) to aligned QF variants using the same `source_id → split` mapping. It measures feasibility, exact reversibility, and distortion only.

It deliberately does **not** use the QF95-trained CNN as a headline detector on QF75/QF85, because a change in detector score could reflect cover-domain shift rather than steganographic detectability. Any future cross-QF detector comparison should use a separately pre-specified domain-generalization protocol.

In [ ]:
RUN_CROSS_SOURCE=False   # set True only if you want the secondary experiment
CROSS_N=500
CROSS_STRATEGIES=['predictability','detectability','joint']
print('RUN_CROSS_SOURCE =',RUN_CROSS_SOURCE)

if RUN_CROSS_SOURCE:
    from rdhlab.dataset import find_bossbase_archive, prepare_bossbase
    raw=Path('/workspace/data/raw')
    archive=find_bossbase_archive(raw)
    primary_map=manifest.assign(source_id=manifest.source_id.astype(str)).set_index('source_id')['split'].to_dict()
    cross_rows=[]

    for variant in config['cross_source']['variants']:
        tag=variant.replace('/','_')
        prepared=prepare_bossbase(
            raw,Path('/workspace/data/processed')/tag,archive,10_000,None,
            int(config['dataset']['split_seed']),source_subdir=variant,allow_lossy_source=True
        )
        vm=pd.read_csv(prepared.manifest_path)
        vm['source_id']=vm.source_id.astype(str)
        vm['split']=vm.source_id.map(primary_map)
        if vm['split'].isna().any():
            raise RuntimeError(f'Could not align all source IDs for {variant}')
        vt=vm[vm.split=='test'].copy()
        # Same source IDs and manifest order across variants.
        vt=vt.set_index('source_id').loc[test.source_id.tolist()].reset_index().head(CROSS_N)

        for i,row in vt.iterrows():
            sid=str(row.source_id); x=read_gray(row.path)
            plans=analyze_blocks(x,bs)
            for strategy in CROSS_STRATEGIES:
                order,br,_=build_strategy_order(x,sid,strategy,local_risk,alpha,bs,seed)
                rr=run_frozen_image_precomputed(x,sid,primary_bpp,strategy,{strategy:order},br,bs,seed,False,None,plans=plans)
                cross_rows.append({
                    'variant':variant,'source_id':sid,'strategy':strategy,
                    'target_net_bpp':primary_bpp,
                    'feasible':bool(rr['feasible']),
                    'exact_image':bool(rr.get('exact_image',False)),
                    'exact_message':bool(rr.get('exact_message',False)),
                    'ber':float(rr.get('ber',np.nan)),
                    'psnr':float(rr.get('psnr',np.nan)),
                    'ssim':float(rr.get('ssim',np.nan)),
                })
        print('cross-source completed:',variant)

    cross=pd.DataFrame(cross_rows)
    cross.to_csv(out/'cross_qf_per_case.csv',index=False)
    cross_summary=(cross.groupby(['variant','strategy'],as_index=False)
        .agg(n=('source_id','size'),feasible_fraction=('feasible','mean'),
             exact_image_fraction=('exact_image','mean'),exact_message_fraction=('exact_message','mean'),
             psnr_mean=('psnr','mean'),ssim_mean=('ssim','mean')))
    cross_summary.to_csv(out/'cross_qf_summary.csv',index=False)
    display(cross_summary)


In [ ]:
# Final machine-readable checkpoint.
resource_complete={
    'status':'COMPLETE',
    'alpha':alpha,
    'primary_benchmark_payload_bpp':primary_bpp,
    'timing_images':int(BENCH_N),
    'timing_cases':int(len(timing)),
    'memory_images_per_strategy':int(MEM_N),
    'feasible_frozen_test_cases':int(recovery['feasible_cases']),
    'exact_image_feasible_cases':int(recovery['exact_image_feasible']),
    'exact_message_feasible_cases':int(recovery['exact_message_feasible']),
    'file_io_feasible_cases':int(recovery['file_io_feasible_cases']),
    'file_io_exact_image_cases':int(recovery['file_io_exact_image']),
    'file_io_exact_message_cases':int(recovery['file_io_exact_message']),
    'metadata_physically_embedded':False,
    'cross_source_ran':bool(RUN_CROSS_SOURCE),
    'no_retuning_permitted':True,
    'allocator_sha256':sha256_file(allocator_path),
    'risk_model_sha256':sha256_file(risk_path),
    'test_source_ids_sha256':complete['test_source_ids_sha256'],
}
atomic_write_json(out/'resource_run_complete.json',resource_complete)
print(json.dumps(resource_complete,indent=2))
print('\nOutputs:',out)


## Reporting notes for the manuscript

- Report **100% exact image and message recovery only over feasible cases**, together with the feasibility rate; do not silently discard infeasible covers.
- State that the current side information is **accounted** using a compact binary model but is not yet physically embedded into a self-contained bitstream. Exact extraction therefore assumes the corresponding metadata is available to the decoder.
- Distinguish the very small blockwise `encode_ms`/`decode_ms` from **end-to-end sender computation**, which includes allocation and block analysis. For the detectability-aware methods, local-risk estimation is expected to dominate.
- Treat the RSS benchmark as a coarse observed process-memory diagnostic, not an exact peak-allocation profiler.
- Do not describe strict RDH as robust to lossy post-embedding transformations unless a separate channel-robust construction is introduced and evaluated.
